# Day 2 動手練習：QKV、Top-k 與 Temperature

這份 notebook 是搭配 [day2] 的動手練習，對應文章裡提到的三個概念：

1. **QKV（Query / Key / Value）**：用 NumPy 手算一次注意力機制，看看「搜尋引擎」的比喻怎麼變成數字運算。
2. **Top-k**：模型只從機率最高的 k 個候選字裡挑答案，改變 k 會發生什麼事。
3. **Temperature**：機率分布被拉平或拉尖，輸出會變得更穩定還是更有變化。

範例句子與模型都跟原始講義不同，模型改用可以在筆電 CPU 上跑的小模型 **Qwen2.5-0.5B-Instruct**（免 API key、不需要登入 Hugging Face），最後附一個選用的 Groq API cell 給想試試雲端小模型的人。

## 環境安裝

只需要 `numpy`、`torch`、`transformers`。第一次執行下面的 cell 會安裝套件；載入模型的 cell 第一次執行會下載約 1GB 的模型檔案，請耐心等待。 執行有問問題，可以把整個檔案給GPT, Claude 或其他 LLM，請他幫你解決，並請求協同執行

```bash

In [1]:
%pip install -q numpy torch transformers accelerate

Note: you may need to restart the kernel to use updated packages.


---
## Part 1｜QKV 自注意力機制手算（NumPy）

沿用 day2.md 裡「搜尋引擎」的比喻：
- **Query**：我想找什麼
- **Key**：有哪些資訊可能相關
- **Value**：真正要取回的內容

這裡不用隨機數字，而是手動指定 4 個中文字的「詞向量」，並用三個簡單的轉換矩陣（教學用，不是訓練出來的）把詞向量轉成 Q、K、V，這樣每一步的數字都可以對照著看。

In [2]:
import numpy as np

np.set_printoptions(precision=2, suppress=True)

# 例句：我 想 學習 AI（4 個 token，每個 token 用 4 維向量代表）
tokens = ["我", "想", "學習", "AI"]

X = np.array([
    [1.0, 0.0, 0.2, 0.0],  # 我
    [0.8, 0.2, 0.0, 0.0],  # 想
    [0.0, 0.2, 1.0, 0.8],  # 學習
    [0.0, 0.0, 0.8, 1.0],  # AI
])

# 教學用的簡化轉換矩陣（不是訓練出來的，只是為了示範 Q/K/V 怎麼從詞向量算出來）
Wq = np.eye(4) * 0.5
Wk = np.eye(4) * 1.0
Wv = np.array([
    [0, 1, 0, 0],
    [1, 0, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0],
], dtype=float)

Q = X @ Wq
K = X @ Wk
V = X @ Wv

print("詞向量 X：\n", X)
print("\nQuery（我想找什麼）：\n", Q)
print("\nKey（有哪些資訊可能相關）：\n", K)
print("\nValue（真正要取回的內容）：\n", V)

詞向量 X：
 [[1.  0.  0.2 0. ]
 [0.8 0.2 0.  0. ]
 [0.  0.2 1.  0.8]
 [0.  0.  0.8 1. ]]

Query（我想找什麼）：
 [[0.5 0.  0.1 0. ]
 [0.4 0.1 0.  0. ]
 [0.  0.1 0.5 0.4]
 [0.  0.  0.4 0.5]]

Key（有哪些資訊可能相關）：
 [[1.  0.  0.2 0. ]
 [0.8 0.2 0.  0. ]
 [0.  0.2 1.  0.8]
 [0.  0.  0.8 1. ]]

Value（真正要取回的內容）：
 [[0.  1.  0.  0.2]
 [0.2 0.8 0.  0. ]
 [0.2 0.  0.8 1. ]
 [0.  0.  1.  0.8]]


接著計算注意力分數：`scores = Q · K^T / sqrt(d_k)`，再做 softmax 得到「每個字對其他字的關注權重」，最後用權重去加總 Value，得到每個字整合上下文後的新表示。

In [3]:
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

d_k = Q.shape[-1]
scores = Q @ K.T / np.sqrt(d_k)
weights = softmax(scores)
output = weights @ V

print("注意力權重（橫列 = 誰在看，縱行 = 看到誰）：\n")
header = "".join([f"{t:<8}" for t in tokens])
print(f"{'':<8}{header}")
for i, row_word in enumerate(tokens):
    row = "".join([f"{v:<8.2f}" for v in weights[i]])
    print(f"{row_word:<8}{row}")

print("\n每個字整合上下文後的新向量（output）：\n", output)

注意力權重（橫列 = 誰在看，縱行 = 看到誰）：

        我       想       學習      AI      
我       0.28    0.26    0.23    0.23    
想       0.28    0.27    0.23    0.23    
學習      0.21    0.20    0.30    0.29    
AI      0.21    0.20    0.30    0.30    

每個字整合上下文後的新向量（output）：
 [[0.1  0.49 0.41 0.46]
 [0.1  0.49 0.41 0.47]
 [0.1  0.37 0.53 0.58]
 [0.1  0.37 0.54 0.58]]


**觀察重點**：「學習」與「AI」的詞向量刻意設計得比較接近，執行後可以看到「學習」對「AI」的注意力權重（約 0.29），明顯高於它對「我」或「想」的權重（約 0.20～0.21），「AI」對「學習」也是同樣的情形。這就是 Attention 在做的事——讓語意相關的字彼此交換更多資訊。

---
## Part 2｜載入小模型，看下一個 token 的機率分布

day2.md 用 transformer-explainer 網站示範了 `Data visualization empowers users to` 這句話。這裡我們換成中文句子，用真正的小模型（Qwen2.5-0.5B-Instruct）算出下一個 token 的機率分布。

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else ("cuda" if torch.cuda.is_available() else "cpu")
)
print(f"使用裝置: {device}")

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
print("正在下載並載入小模型（第一次執行需要幾分鐘）...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()
print("載入完成！")

使用裝置: mps
正在下載並載入小模型（第一次執行需要幾分鐘）...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

載入完成！


In [5]:
prompt = "人工智慧未來會"

inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits[0, -1]  # 最後一個位置的 logits，代表「下一個 token」的分數

probs = torch.softmax(logits, dim=-1)

top_k = 5
top_probs, top_ids = torch.topk(probs, top_k)

print(f"輸入句子: 「{prompt}」")
print(f"模型判斷，接在後面機率最高的 {top_k} 個 token：\n")
for prob, tid in zip(top_probs, top_ids):
    token_text = tokenizer.decode([tid])
    print(f"  {token_text!r:<10} (ID: {tid.item():<6})  機率: {prob.item() * 100:5.2f}%")

輸入句子: 「人工智慧未來會」
模型判斷，接在後面機率最高的 5 個 token：

  '如何'       (ID: 100007)  機率: 22.36%
  '成為'       (ID: 106027)  機率:  3.66%
  '帶來'       (ID: 113089)  機率:  3.66%
  '是'        (ID: 20412 )  機率:  3.44%
  '取代'       (ID: 107806)  機率:  3.03%


---
## Part 3｜調整 Top-k

day2.md 把 Top-k 從 5 改成 2，候選字變少、機率會在剩下的候選字之間重新分配。我們也做一次同樣的實驗。

In [6]:
def show_topk(probs, k, tokenizer):
    top_probs, top_ids = torch.topk(probs, k)
    renorm = top_probs / top_probs.sum()  # 只在候選字之間重新分配機率
    print(f"Top-k = {k}")
    for p, r, tid in zip(top_probs, renorm, top_ids):
        token_text = tokenizer.decode([tid])
        print(f"  {token_text!r:<10} 原始機率: {p.item() * 100:5.2f}%   重新分配後: {r.item() * 100:5.2f}%")
    print()

show_topk(probs, 5, tokenizer)
show_topk(probs, 2, tokenizer)

Top-k = 5
  '如何'       原始機率: 22.36%   重新分配後: 61.72%
  '成為'       原始機率:  3.66%   重新分配後: 10.16%
  '帶來'       原始機率:  3.66%   重新分配後: 10.16%
  '是'        原始機率:  3.44%   重新分配後:  9.52%
  '取代'       原始機率:  3.03%   重新分配後:  8.40%

Top-k = 2
  '如何'       原始機率: 22.36%   重新分配後: 85.94%
  '成為'       原始機率:  3.66%   重新分配後: 14.06%



**觀察重點**：Top-k 只是先「砍候選名單」，砍完之後才在剩下的候選字裡重新分配機率——k 越小，模型的選擇越保守。

---
## Part 4｜調整 Temperature

Temperature 是在 softmax 之前，先把 logits 除以溫度值：`softmax(logits / T)`。

- T 小於 1：機率分布被拉尖，最高機率的字更容易被選中，輸出更穩定。
- T 大於 1：機率分布被拉平，低機率的字也有機會被選中，輸出更有變化。

先用同一個 prompt 的 logits，比較低、中、高三種溫度下的 Top-5 機率。

In [7]:
def logits_to_topk(logits, temperature, k, tokenizer):
    scaled = logits / temperature
    probs_t = torch.softmax(scaled, dim=-1)
    top_probs, top_ids = torch.topk(probs_t, k)
    print(f"Temperature = {temperature}")
    for p, tid in zip(top_probs, top_ids):
        token_text = tokenizer.decode([tid])
        print(f"  {token_text!r:<10} 機率: {p.item() * 100:5.2f}%")
    print()

for t in [0.3, 1.0, 1.5]:
    logits_to_topk(logits, t, 5, tokenizer)

Temperature = 0.3
  '如何'       機率: 98.83%
  '是'        機率:  0.19%
  '成為'       機率:  0.19%
  '帶來'       機率:  0.19%
  '取代'       機率:  0.12%

Temperature = 1.0
  '如何'       機率: 22.36%
  '成為'       機率:  3.66%
  '帶來'       機率:  3.66%
  '是'        機率:  3.44%
  '取代'       機率:  3.03%

Temperature = 1.5
  '如何'       機率:  4.08%
  '成為'       機率:  1.25%
  '帶來'       機率:  1.25%
  '是'        機率:  1.17%
  '取代'       機率:  1.10%



接著實際生成幾次文字，感受一下低溫和高溫在真實輸出上的差異：低溫的幾次輸出應該長得很像，高溫的幾次輸出則會明顯不同。

In [8]:
def sample_multiple(prompt, temperature, n=3, max_new_tokens=12):
    print(f"Temperature = {temperature}")
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    for i in range(n):
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_k=50,
            )
        text = tokenizer.decode(
            output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )
        print(f"  第 {i + 1} 次輸出: {text}")
    print()

sample_multiple(prompt, temperature=0.3)
sample_multiple(prompt, temperature=1.5)

Temperature = 0.3
  第 1 次輸出: 如何影響我們的生活？請以「智能手機」為
  第 2 次輸出: 如何影響我們的生活？ 人工智能（AI）的發展
  第 3 次輸出: 如何影響我們的生活？請以「AI在生活中的

Temperature = 1.5
  第 1 次輸出: 影響人類什麼方向?
我們認為，人工智能有三個可能
  第 2 次輸出: 如何影響我們的日常生活？

智能科技將對我日常
  第 3 次輸出: 發展在哪種形式下，哪些行業可能會因此獲得較



---
## （選用）改用 Groq API 跑雲端小模型

如果不想在本機下載模型，也可以改用 [Groq](https://console.groq.com/) 的雲端 API 跑一個小模型。步驟：

1. 到 https://console.groq.com/keys 申請一組 API Key。
2. 在終端機設定環境變數（不要把金鑰寫死在程式碼或 notebook 裡）：
   ```bash
   export GROQ_API_KEY="你的金鑰"
   ```
3. 重新啟動 Jupyter，讓環境變數生效，再執行下面的 cell。

> 注意：Groq 的 Chat Completions API 目前只開放 `temperature` 和 `top_p` 兩個參數，沒有 `top_k`，所以 Top-k 的實驗還是要用 Part 2～3 的地端模型。

In [ ]:
import os

groq_api_key = os.environ.get("GROQ_API_KEY")

if not groq_api_key:
    print("尚未設定 GROQ_API_KEY，略過這個 cell。")
else:
    try:
        from groq import Groq
    except ImportError:
        import sys, subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "groq"], check=True)
        from groq import Groq

    client = Groq(api_key=groq_api_key)

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=1.0,
        max_tokens=20,
        top_p=1,
    )
    print(response.choices[0].message.content)

---
## 停下來想一想

呼應 day2.md 的結語：模型輸出讀起來流暢，不代表內容一定正確。

- 在 Part 4 的高溫實驗裡，有沒有出現讀起來通順、但其實內容怪怪的句子？
- 如果要讓一個客服機器人的回答更「穩定」，你會選擇比較低還是比較高的 Temperature？為什麼？
- Top-k 和 Temperature 可以同時調整，兩者一起使用時，你覺得誰的影響力比較大？